# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/octane21/flyrank-internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

print("Secret successfully loaded!")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    'dim_clients':f"read_parquet('{rel}/dim_clients.parquet')",
    'dim_content':f"read_parquet('{rel}/dim_content.parquet')",
    'fact_content_daily_performance':f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')",
    'fact_content_query_90d':f"read_parquet('{rel}/fact_content_query_90d.parquet')",
}


Secret successfully loaded!


In [2]:
df = con.sql(f"""
WITH content_id_hv_criteria AS (
    SELECT
        content_hash_id
    FROM {TABLES['fact_content_daily_performance']}
    WHERE
        gsc_data_available IS TRUE
        AND ga4_data_available IS NOT NULL
    GROUP BY content_hash_id
    HAVING
        COUNT(DISTINCT report_date) >= 120
        AND
        DATE_DIFF(
            'day',
            MIN(report_date),
            MAX(report_date)
        ) + 1 = COUNT(DISTINCT report_date)
)

SELECT
      report_date,
      client_hash_id,
      fact.content_hash_id,
      ga4_data_available,
      gsc_data_available,
      gsc_impressions,
      gsc_clicks,
      gsc_avg_position,
      ga4_pageviews,
      ga4_sessions,
      ga4_users,
      ga4_engaged_sessions,
      sessions_ai,
      sessions_organic,
      scroll_events

FROM {TABLES['fact_content_daily_performance']} AS fact
INNER JOIN content_id_hv_criteria AS criteria
    ON fact.content_hash_id = criteria.content_hash_id
WHERE
    fact.gsc_data_available IS TRUE
    AND fact.ga4_data_available IS NOT NULL;
""").df()

df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,ga4_data_available,gsc_data_available,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,sessions_ai,sessions_organic,scroll_events
0,2025-02-10,client_9958f0a7ae1df715,content_92fd77e367253da2,False,True,4,0,7.250000,0,0,0,0,0,0,0
1,2025-02-10,client_9958f0a7ae1df715,content_00dafe96928080fb,False,True,18,0,4.888889,0,0,0,0,0,0,0
2,2025-02-10,client_9958f0a7ae1df715,content_7a1bdd56e533fef5,False,True,14,0,8.714286,0,0,0,0,0,0,0
3,2025-02-10,client_9958f0a7ae1df715,content_d73205f6d972a269,False,True,33,3,3.848485,0,0,0,0,0,0,0
4,2025-02-10,client_9958f0a7ae1df715,content_c936b727ae06f104,False,True,18,0,14.333333,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2562564,2026-06-30,client_65de48885f4ef01b,content_f862eedcac7d187e,False,True,25,0,8.040000,0,0,0,0,0,0,0
2562565,2026-06-30,client_65de48885f4ef01b,content_153da632dd6adb12,False,True,44,0,8.068182,0,0,0,0,0,0,0
2562566,2026-06-30,client_65de48885f4ef01b,content_4e0c7263aa6aab62,False,True,63,1,5.555556,0,0,0,0,0,0,0
2562567,2026-06-30,client_65de48885f4ef01b,content_fa9131f1743ae0ae,False,True,73,0,5.150685,0,0,0,0,0,0,0


In [3]:
result = df.groupby('content_hash_id').agg(start_date = ('report_date', "min"),end_date = ('report_date', "max")).sort_values(['start_date','end_date'], ascending=[False,True])
result

,start_date,end_date
content_hash_id,,
content_0060ca245492494f,2026-03-03,2026-06-30
content_00823406ee06d1e7,2026-03-03,2026-06-30
content_0084ee8efa2fcd23,2026-03-03,2026-06-30
content_00e0da358d5df2e7,2026-03-03,2026-06-30
content_00efcc9c1c9bba3d,2026-03-03,2026-06-30
...,...,...
content_9987be00b8c14a02,2025-01-27,2026-06-30
content_cf0500bbb61cc8b9,2025-01-27,2026-06-30
content_d02be57d816cf3d7,2025-01-27,2026-06-30


In [4]:
import pandas as pd

In [5]:
incomplete_content_month = result.copy().reset_index()

incomplete_content_month['incomplete_start_month'] = (
    incomplete_content_month['start_date'].dt.day > 15
)

incomplete_content_month['incomplete_end_month'] = (
    incomplete_content_month['end_date'].dt.day < 15
)

incomplete_content_month['start_month'] = (
    incomplete_content_month['start_date'].dt.to_period('M')
)

incomplete_content_month['end_month'] = (
    incomplete_content_month['end_date'].dt.to_period('M')
)

df['month'] = df['report_date'].dt.to_period('M')

data_complete_month = df.merge(
    incomplete_content_month[
        [
            'content_hash_id',
            'start_month',
            'end_month',
            'incomplete_start_month',
            'incomplete_end_month'
        ]
    ],
    on='content_hash_id',
    how='left'
)

data_complete_month = data_complete_month[
    ~(
        (
            data_complete_month['incomplete_start_month'] &
            (data_complete_month['month'] == data_complete_month['start_month'])
        ) |
        (
            data_complete_month['incomplete_end_month'] &
            (data_complete_month['month'] == data_complete_month['end_month'])
        )
    )
].copy()

In [6]:
data_complete_month.groupby('content_hash_id').agg(days = ('report_date', "count")).sort_values('days',ascending=True)

,days
content_hash_id,
content_eb25f969a5734bc1,109
content_9d0a8af767ca01f3,110
content_93cc7359119173d8,110
content_27283b2c07a61c28,110
content_8f3e71fbfe536666,110
...,...
content_1c7ae2a4d8295008,515
content_f3a75d8cf58dd50b,515
content_84f5a9ecfefa108e,515


In [7]:
data_complete_month[data_complete_month['content_hash_id'] == 'content_f3ba8c1f94eb36f5']

,report_date,client_hash_id,content_hash_id,ga4_data_available,gsc_data_available,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,sessions_ai,sessions_organic,scroll_events,month,start_month,end_month,incomplete_start_month,incomplete_end_month
7309,2025-05-25,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,False,True,15,0,22.000000,0,0,0,0,0,0,0,2025-05,2025-05,2026-06,False,False
7963,2025-05-15,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,False,True,22,0,17.954545,0,0,0,0,0,0,0,2025-05,2025-05,2026-06,False,False
8156,2025-05-20,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,False,True,18,0,11.888889,0,0,0,0,0,0,0,2025-05,2025-05,2026-06,False,False
8699,2025-05-12,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,False,True,7,1,4.000000,0,0,0,0,0,0,0,2025-05,2025-05,2026-06,False,False
8902,2025-05-22,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,False,True,19,0,12.631579,0,0,0,0,0,0,0,2025-05,2025-05,2026-06,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2453839,2026-06-10,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,False,True,5,0,16.000000,0,0,0,0,0,0,0,2026-06,2025-05,2026-06,False,False
2492460,2026-06-13,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,False,True,9,0,18.555556,0,0,0,0,0,0,0,2026-06,2025-05,2026-06,False,False
2492848,2026-06-20,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,False,True,2,0,20.000000,0,0,0,0,0,0,0,2026-06,2025-05,2026-06,False,False
2557179,2026-06-23,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,False,True,7,0,18.714286,0,0,0,0,0,0,0,2026-06,2025-05,2026-06,False,False


In [8]:
data_complete_month.drop(['start_month', 'end_month','incomplete_start_month','incomplete_end_month'], axis=1, inplace=True)

In [9]:
grouped = data_complete_month.groupby([ 'client_hash_id','content_hash_id', 'month'])

numeric_cols = data_complete_month.select_dtypes(include='number').columns.drop('gsc_avg_position')
bool_cols = data_complete_month.select_dtypes(include='bool').columns

content_month = grouped.agg({
    **{col: 'sum' for col in numeric_cols},
    **{col: 'any' for col in bool_cols},
    'gsc_avg_position':'mean'
}).reset_index()

rename_dict = {
    col: f"{col}_30d" for col in numeric_cols.append(pd.Index(['gsc_avg_position']))
}
content_month = content_month.rename(columns=rename_dict)

In [10]:
content_month

,client_hash_id,content_hash_id,month,gsc_impressions_30d,gsc_clicks_30d,ga4_pageviews_30d,ga4_sessions_30d,ga4_users_30d,ga4_engaged_sessions_30d,sessions_ai_30d,sessions_organic_30d,scroll_events_30d,ga4_data_available,gsc_data_available,gsc_avg_position_30d
0,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-03,330,1,0,0,0,0,0,0,0,False,True,7.675690
1,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-04,393,5,5,5,4,0,0,5,0,True,True,7.441862
2,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-05,872,1,1,1,1,0,0,1,0,True,True,7.100545
3,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-06,1437,6,8,8,8,1,0,6,1,True,True,6.115233
4,client_0fa64a184f18a4a0,content_003959d7e5063269,2026-03,464,1,1,1,1,0,0,1,0,True,True,8.768573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85139,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-02,939,4,5,5,5,0,0,3,0,True,True,8.860769
85140,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-03,1534,3,6,6,6,0,0,4,0,True,True,9.573120
85141,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-04,482,0,4,4,3,0,0,0,0,True,True,20.424837
85142,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-05,269,0,5,5,5,1,0,0,1,True,True,32.566968


In [11]:
content_month.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85144 entries, 0 to 85143
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype    
---  ------                    --------------  -----    
 0   client_hash_id            85144 non-null  object   
 1   content_hash_id           85144 non-null  object   
 2   month                     85144 non-null  period[M]
 3   gsc_impressions_30d       85144 non-null  int64    
 4   gsc_clicks_30d            85144 non-null  int64    
 5   ga4_pageviews_30d         85144 non-null  int64    
 6   ga4_sessions_30d          85144 non-null  int64    
 7   ga4_users_30d             85144 non-null  int64    
 8   ga4_engaged_sessions_30d  85144 non-null  int64    
 9   sessions_ai_30d           85144 non-null  int64    
 10  sessions_organic_30d      85144 non-null  int64    
 11  scroll_events_30d         85144 non-null  int64    
 12  ga4_data_available        85144 non-null  bool     
 13  gsc_data_available        85144

In [12]:
cols = content_month.columns.drop(['client_hash_id', 'content_hash_id', 'month', 'gsc_data_available', 'ga4_data_available'])
cols

Index(['gsc_impressions_30d', 'gsc_clicks_30d', 'ga4_pageviews_30d',
       'ga4_sessions_30d', 'ga4_users_30d', 'ga4_engaged_sessions_30d',
       'sessions_ai_30d', 'sessions_organic_30d', 'scroll_events_30d',
       'gsc_avg_position_30d'],
      dtype='object')

In [13]:
content_month = content_month.sort_values(
    ['client_hash_id', 'content_hash_id', 'month']
).copy()

grouped = content_month.groupby(
    ['client_hash_id', 'content_hash_id']
)

for col in cols:
    base_col = col[:-4]

    content_month[f'{base_col}_prev_30d'] = grouped[col].shift(1)
    content_month[f'{base_col}_2m_ago_30d'] = grouped[col].shift(2)
    content_month[f'{base_col}_future_30d'] = grouped[col].shift(-1)

In [14]:
content_month[content_month['content_hash_id'] == 'content_f3baf9504688']

,client_hash_id,content_hash_id,month,gsc_impressions_30d,gsc_clicks_30d,ga4_pageviews_30d,ga4_sessions_30d,ga4_users_30d,ga4_engaged_sessions_30d,sessions_ai_30d,...,sessions_ai_future_30d,sessions_organic_prev_30d,sessions_organic_2m_ago_30d,sessions_organic_future_30d,scroll_events_prev_30d,scroll_events_2m_ago_30d,scroll_events_future_30d,gsc_avg_position_prev_30d,gsc_avg_position_2m_ago_30d,gsc_avg_position_future_30d


In [25]:
dataset = content_month[
    content_month['gsc_impressions_2m_ago_30d'].notna() &
    content_month['gsc_impressions_future_30d'].notna()
].copy()

In [26]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45019 entries, 2 to 85142
Data columns (total 45 columns):
 #   Column                           Non-Null Count  Dtype    
---  ------                           --------------  -----    
 0   client_hash_id                   45019 non-null  object   
 1   content_hash_id                  45019 non-null  object   
 2   month                            45019 non-null  period[M]
 3   gsc_impressions_30d              45019 non-null  int64    
 4   gsc_clicks_30d                   45019 non-null  int64    
 5   ga4_pageviews_30d                45019 non-null  int64    
 6   ga4_sessions_30d                 45019 non-null  int64    
 7   ga4_users_30d                    45019 non-null  int64    
 8   ga4_engaged_sessions_30d         45019 non-null  int64    
 9   sessions_ai_30d                  45019 non-null  int64    
 10  sessions_organic_30d             45019 non-null  int64    
 11  scroll_events_30d                45019 non-null  int64    


In [27]:
contents = con.sql(f"""
SELECT
    content_hash_id,
    content_created_date,
    content_updated_date,
    content_type,
    search_volume,
    competition,
    competition_level,
    cpc,
    main_intent,
    backlinks,
    category_count,
    word_count,
    char_count
FROM {TABLES['dim_content']}
""").df()

contents

,content_hash_id,content_created_date,content_updated_date,content_type,search_volume,competition,competition_level,cpc,main_intent,backlinks,category_count,word_count,char_count
0,content_004de9653278b5a4,2026-05-30,2026-07-01,keyword article,30,0.91,HIGH,0.98,transactional,16,3,2555,15682
1,content_00dc5efae381b2ab,2026-06-12,2026-07-01,keyword article,10,0.00,LOW,0.00,commercial,0,4,2430,15438
2,content_01410f2556c327ac,2026-05-09,2026-07-01,keyword article,480,0.36,MEDIUM,0.62,informational,169,4,2645,16576
3,content_019f27f634053ca7,2026-06-15,2026-06-15,keyword article,0,0.00,LOW,0.00,transactional,0,4,2522,15457
4,content_01efa71faea45dcc,2026-05-21,2026-06-01,keyword article,2400,0.70,HIGH,0.90,transactional,52,4,2552,15776
...,...,...,...,...,...,...,...,...,...,...,...,...,...
519601,content_9643d3c771bd23bd,2026-03-06,2026-06-17,keyword article,10,0.49,MEDIUM,0.02,commercial,0,3,1492,9694
519602,content_9646b5037272bf2b,2025-12-17,2026-06-17,keyword article,0,0.00,LOW,0.00,informational,0,0,1403,9005
519603,content_964d01d55c1d3beb,2025-08-03,2026-06-17,keyword article,20,0.93,HIGH,8.56,transactional,<NA>,0,1491,9359
519604,content_965127aa4bcd558d,2025-08-03,2026-06-17,keyword article,10,0.33,MEDIUM,0.65,informational,<NA>,0,1446,8737


In [28]:
contents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 519606 entries, 0 to 519605
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   content_hash_id       519606 non-null  object        
 1   content_created_date  519606 non-null  datetime64[us]
 2   content_updated_date  519606 non-null  datetime64[us]
 3   content_type          519606 non-null  object        
 4   search_volume         376984 non-null  Int64         
 5   competition           376984 non-null  float64       
 6   competition_level     375150 non-null  object        
 7   cpc                   376984 non-null  float64       
 8   main_intent           371208 non-null  object        
 9   backlinks             252132 non-null  Int64         
 10  category_count        519606 non-null  int64         
 11  word_count            341838 non-null  Int64         
 12  char_count            341838 non-null  Int64         
dtyp

In [29]:
dataset = dataset.merge(contents, on='content_hash_id', how='inner')
dataset

,client_hash_id,content_hash_id,month,gsc_impressions_30d,gsc_clicks_30d,ga4_pageviews_30d,ga4_sessions_30d,ga4_users_30d,ga4_engaged_sessions_30d,sessions_ai_30d,...,content_type,search_volume,competition,competition_level,cpc,main_intent,backlinks,category_count,word_count,char_count
0,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-05,872,1,1,1,1,0,0,...,keyword article,10,0.00,LOW,0.00,transactional,<NA>,0,1508,9976
1,client_0fa64a184f18a4a0,content_003959d7e5063269,2026-05,522,3,7,4,4,0,0,...,keyword article,20,1.00,HIGH,0.29,transactional,0,3,2892,21511
2,client_0fa64a184f18a4a0,content_04875f00881d45ed,2026-05,777,1,1,1,1,0,0,...,keyword article,50,0.03,LOW,0.00,commercial,49,9,2874,20710
3,client_0fa64a184f18a4a0,content_04b7abe75a1b84f7,2026-05,17485,34,27,26,26,4,0,...,keyword article,10,0.82,HIGH,1.97,transactional,0,4,2533,18492
4,client_0fa64a184f18a4a0,content_092c998a5c931d11,2026-05,1768,1,1,1,1,0,0,...,keyword article,20,0.90,HIGH,0.00,transactional,38,5,2931,21104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45014,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-01,1535,4,4,4,4,0,0,...,keyword article,260,0.04,LOW,0.03,transactional,<NA>,0,<NA>,<NA>
45015,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-02,939,4,5,5,5,0,0,...,keyword article,260,0.04,LOW,0.03,transactional,<NA>,0,<NA>,<NA>
45016,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-03,1534,3,6,6,6,0,0,...,keyword article,260,0.04,LOW,0.03,transactional,<NA>,0,<NA>,<NA>
45017,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-04,482,0,4,4,3,0,0,...,keyword article,260,0.04,LOW,0.03,transactional,<NA>,0,<NA>,<NA>


In [31]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45019 entries, 0 to 45018
Data columns (total 57 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   client_hash_id                   45019 non-null  object        
 1   content_hash_id                  45019 non-null  object        
 2   month                            45019 non-null  period[M]     
 3   gsc_impressions_30d              45019 non-null  int64         
 4   gsc_clicks_30d                   45019 non-null  int64         
 5   ga4_pageviews_30d                45019 non-null  int64         
 6   ga4_sessions_30d                 45019 non-null  int64         
 7   ga4_users_30d                    45019 non-null  int64         
 8   ga4_engaged_sessions_30d         45019 non-null  int64         
 9   sessions_ai_30d                  45019 non-null  int64         
 10  sessions_organic_30d             45019 non-null  int64    

In [ ]:
dataset.to_csv("dataset_flyrank")

In [ ]:
feature_cols = [
    'ga4_data_available',
    'ga4_data_available'
    'gsc_impressions',
    'gsc_clicks',
    'gsc_avg_position',
    'ga4_pageviews',
    'ga4_sessions',
    'ga4_users',
    'ga4_engaged_sessions',
    'sessions_ai',
    'sessions_organic',
    'scroll_events',

    'content_created_date',
    'content_updated_date',
    'content_type',
    'search_volume',
    'competition',
    'competition_level',
    'cpc',
    'main_intent',
    'backlinks',
    'category_count',
    'word_count',
    'char_count',
]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.